# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row in the final modeling frame represents **one content item for one pseudonymized client** (`client_hash_id` × `content_hash_id`).

The source table, `fact_content_daily_performance`, is at daily grain: one row per report date, client, and content item.

### Time windows

- **February 2026:** feature window — information available through 2026-02-28.
- **March 2026:** label window — the subsequent observed outcome.

The two windows are deliberately separated. Features must be based only on information available by the end of February, while the March outcome is used as the label. This prevents future information from entering the decision-time features.

In [3]:
import duckdb
from google.colab import userdata

# Get the Hugging Face token securely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# Time windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to the FlyRank warehouse.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected to the FlyRank warehouse.
Feature window: February 2026
Label window: March 2026


In [4]:
# Query 1 — verify the daily grain

grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_daily_keys
FROM {FEB}
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_daily_keys
0,7355108,7355108


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields

The feature fields will come only from the February 2026 feature window and will be available by the decision cutoff at the end of February.

- `gsc_impressions` — search exposure observed in February.
- `gsc_clicks` — search clicks observed in February.
- `gsc_sum_position` — search-position information observed in February.
- `ga4_sessions` — analytics sessions observed in February.
- `content_age_days` — content age known by the end of February.

### Label

`went_dark` is the March 2026 outcome: 1 when the content item records zero GSC clicks in March, otherwise 0. It is an observed outcome used as a proxy for prioritising review, not a guarantee that refreshing the page will improve performance.

### Context fields

`client_hash_id`, `content_hash_id`, and the report date identify the observation and help with grouping, joining, and verification. They are context/keys rather than predictive features.

### Excluded fields

I exclude March/future performance fields, fields derived from the March outcome, and product-decision fields such as health or priority flags. These would not be available at the decision moment or could leak the outcome into the features.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 2 — verify the February 2026 slice

feb_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}
""").df()

feb_check


,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 3 — verify data availability using IS TRUE

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS available_rows,
    MIN(report_date) AS first_available_date,
    MAX(report_date) AS last_available_date
FROM {FEB}
WHERE gsc_data_available IS TRUE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows,first_available_date,last_available_date
0,2621783,2026-02-01,2026-02-28


In [7]:
# Inspect the available warehouse fields

con.sql(f"""
DESCRIBE
SELECT *
FROM {FEB}
LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Five-feature frame

I use five features from the February 2026 feature window:

1. **gsc_impressions** — knowable at the decision moment because February search impressions have already been observed by the end of February.
2. **gsc_clicks** — knowable at the decision moment because February search clicks have already been observed by the end of February.
3. **gsc_avg_position** — knowable at the decision moment because the observed February search-position information is available before the March outcome window.
4. **ga4_pageviews** — knowable at the decision moment because February pageviews have already been recorded in the analytics data.
5. **ga4_sessions** — knowable at the decision moment because February sessions have already been recorded in the analytics data.

I do not use March performance information as a feature because March is the outcome window.

In [8]:
# Build the five-feature frame from February 2026

features_feb = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_sessions) AS ga4_sessions
FROM {FEB}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Feature frame shape:", features_feb.shape)
features_feb.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (321546, 7)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,6.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,1.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,9.0,6.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,3.0


### Deliberate leakage check

I intentionally add a label-derived field to demonstrate leakage. Because this field contains information derived from the outcome, a model can appear unrealistically strong. I then remove the leaked field and keep the honest feature set for decision-time use.

In [9]:
# Deliberate leakage demonstration

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create the March outcome: 1 = zero GSC clicks, 0 = otherwise.
march_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 1
        ELSE 0
    END AS went_dark
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

# Join the March outcome to the February feature frame.
model_frame = features_feb.merge(
    march_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Honest features
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

X = model_frame[feature_cols].fillna(0)
y = model_frame["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Honest model
honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_model.predict(X_test))

# Deliberate leakage: put the label itself into the features
X_leaked = model_frame[feature_cols + ["went_dark"]].fillna(0)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaked, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaked_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaked_model.fit(Xl_train, yl_train)
leaked_score = accuracy_score(
    yl_test,
    leaked_model.predict(Xl_test)
)

print(f"Honest feature score: {honest_score:.4f}")
print(f"Leaked feature score: {leaked_score:.4f}")
print("The leaked feature is removed from the final feature set.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest feature score: 0.8187
Leaked feature score: 1.0000
The leaked feature is removed from the final feature set.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Data limits

This data can support observed, directional decision-support signals, but it cannot tell us with certainty whether refreshing a page will cause its performance to improve.

The available history can also be unbalanced across clients and data sources. Some observations may have GSC-only or GA4-only availability, so missingness and source availability must be checked before interpreting a feature.

The February feature window and March label window are adjacent and may contain overlapping real-world effects such as seasonality, external events, or changes in search behaviour. Therefore, the observed March outcome should not be interpreted as proof of causation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.